# FlashAttention: IO-Aware Attention - 실습 코드 2: FlashAttention Tiling 원리 구현 (NumPy)

- Tutorial ID: `expand-flash-attention`
- Tutorial: FlashAttention: IO-Aware Attention
- Section ID: `expand-flash-attention-code-2`
- Section: 실습 코드 2: FlashAttention Tiling 원리 구현 (NumPy)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: FlashAttention Tiling 원리 구현 (NumPy)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def flash_attention_tiled(Q, K, V, block_size=64):
    """FlashAttention의 Tiling 알고리즘을 NumPy로 구현
    
    핵심: 전체 N×N Attention 행렬을 저장하지 않고
    작은 블록 단위로 처리하여 메모리 O(N²) → O(N)
    """
    N, d = Q.shape
    scale = np.sqrt(d)
    
    # 최종 출력과 통계값
    O = np.zeros_like(Q)        # (N, d)
    l = np.zeros(N)             # 각 행의 softmax 분모 (누적)
    m = np.full(N, -np.inf)     # 각 행의 최대값 (누적)
    
    # 블록 단위 처리 (Tiling)
        for j in range(0, N, block_size):
        j_end = min(j + block_size, N)
        
        # 현재 블록의 K, V
        K_block = K[j:j_end]    # (block, d)
        V_block = V[j:j_end]    # (block, d)
        
        # 각 쿼리 행에 대해 현재 블록과의 attention 계산
                for i in range(0, N, block_size):
            i_end = min(i + block_size, N)
            
            Q_block = Q[i:i_end]  # (block, d)
            
            # S_block = Q_block @ K_block^T / sqrt(d)
            S_block = Q_block @ K_block.T / scale  # (block_i, block_j)
            
            # Online Softmax 갱신
            m_new = np.maximum(m[i:i_end, None], S_block.max(axis=-1, keepdims=True))
            
            # 이전 분모를 새 최대값으로 보정
            l_old = l[i:i_end, None] * np.exp(m[i:i_end, None] - m_new)
            
            # 현재 블록의 softmax 분모
            P_block = np.exp(S_block - m_new)  # 수치 안정적 softmax
            l_new = l_old + P_block.sum(axis=-1, keepdims=True)
            
            # 이전 출력을 새 최대값으로 보정
            O_old = O[i:i_end] * (l[i:i_end, None] * np.exp(m[i:i_end, None] - m_new)) / l_new
            
            # 현재 블록의 기여분
            O_new = (P_block @ V_block) / l_new
            
            # 누적 출력
            O[i:i_end] = O_old + O_new
            
            # 통계값 갱신
            m[i:i_end] = m_new.squeeze()
            l[i:i_end] = l_new.squeeze()
    
    return O

# ── 표준 Attention과 비교 ──
def standard_attention(Q, K, V):
    """표준 Attention (메모리 O(N²))"""
    d = Q.shape[-1]
        scores = Q @ K.T / np.sqrt(d)
    # 수치 안정적 softmax
    scores_max = scores.max(axis=-1, keepdims=True)
        exp_scores = np.exp(scores - scores_max)
        weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    return weights @ V

# 테스트
np.random.seed(42)
N, d = 256, 64
Q = np.random.randn(N, d) * 0.1
K = np.random.randn(N, d) * 0.1
V = np.random.randn(N, d) * 0.1

# 두 구현 비교
O_standard = standard_attention(Q, K, V)
O_flash = flash_attention_tiled(Q, K, V, block_size=32)

# 결과 일치 확인
max_diff = np.abs(O_standard - O_flash).max()
mean_diff = np.abs(O_standard - O_flash).mean()
print(f"=== FlashAttention Tiling 검증 ===")
print(f"Max 차이: {max_diff:.2e}")
print(f"Mean 차이: {mean_diff:.2e}")
print(f"일치 여부: {'✅ PASS' if max_diff < 1e-5 else '❌ FAIL'}")

# 메모리 비교
standard_mem = N * N * 8  # FP64 Attention matrix
flash_mem = N * d * 8 * 3  # Q, K, V만
print(f"\n메모리 사용:")
print(f"  Standard: {standard_mem/1e6:.2f} MB (Attention 행렬)")
print(f"  FlashAttention: {flash_mem/1e6:.2f} MB (Attention 행렬 불필요)")
print(f"  절감률: {(1 - flash_mem/standard_mem)*100:.1f}%")